# BLP

(MGTECON 607 HW 4) Consider $T=500$ independent markets indexed by $t=1,\dots,T$. Each market has $J=2$ inside goods, indexed by $j\in\{1,2\}$, and an outside option normalized to zero utility.

For each market $t$ draw:
\begin{align*}
x_{jt} &\overset{\mathrm{i.i.d.}}{\sim} \mathcal{N}(0,1) &&\text{(observed characteristic, scalar)}\\
z_{jt} &\overset{\mathrm{i.i.d.}}{\sim} \mathcal{N}(0,1) &&\text{(excluded instrument, scalar)}\\
\eta_{jt} &\overset{\mathrm{i.i.d.}}{\sim} \mathcal{N}(0,1) &&\text{(price shock)}\\
\xi_t &=
\begin{bmatrix}\xi_{1t}\\ \xi_{2t}\end{bmatrix}
\sim
\mathcal{N}\!\left(0,\;
\begin{bmatrix}
1 & 0.5\\
0.5 & 1
\end{bmatrix}\right)
&&\text{(unobserved quality, correlated across products).}
\end{align*}
Set prices by
$$
p_{jt}=\exp\!\Big(0.5\,\xi_{jt}+\sqrt{1-0.5^2}\,\eta_{jt}+z_{jt}\Big).
$$
Consumer $i$ in market $t$ has random price sensitivity
$$
\alpha_i \sim \mathcal{N}(\mu,\sigma^2),
$$
a scalar that loads on price for both goods. Utility is
$$
U_{ijt}=\beta x_{jt}-\alpha_i p_{jt}+\xi_{jt}+\epsilon_{ijt}, \qquad \epsilon_{ijt} \overset{\mathrm{i.i.d.}}{\sim} \mathrm{Gumbel},
$$
with the outside option normalized so its utility is $\epsilon_{i0t}$.

Market shares $y_{jt}$ are the logit choice probabilities averaged over $\alpha_i$. For data generation, approximate the integral over $\alpha$ using $M_{\text{true}}=5000$ simulation draws.

The unknown parameter vector is $\theta=(\mu,\sigma,\beta)$. In the Monte Carlo, the true values are $\mu=2.0$, $\sigma=0.5$, and $\beta=0.5$.

In [1]:
import numpy as np
import pandas as pd
from scipy.special import logsumexp
from scipy.optimize import minimize

rng = np.random.default_rng(607)

### Data Simulation

In [2]:
T = 500
J = 2
mu_true = 2.0
sigma_true = 0.5
beta_true = 0.5
M_true = 5000

x = rng.normal(0.0, 1.0, size=(T, J))
z = rng.normal(0.0, 1.0, size=(T, J))
eta = rng.normal(0.0, 1.0, size=(T, J))

cov_xi = np.array([[1.0, 0.5], [0.5, 1.0]])
xi = rng.multivariate_normal(mean=np.zeros(J), cov=cov_xi, size=T)

p = np.exp(0.5 * xi + np.sqrt(1 - 0.5**2) * eta + z)

W_true = rng.normal(0.0, 1.0, size=M_true)
alpha_true = mu_true + sigma_true * W_true

# Vectorized data-generation shares: shape (T, J)
v_true = beta_true * x + xi
U_true = v_true[None, :, :] - alpha_true[:, None, None] * p[None, :, :]  # (M_true, T, J)
denom_true = 1.0 + np.exp(logsumexp(U_true, axis=2))
probs_true = np.exp(U_true) / denom_true[:, :, None]
y = probs_true.mean(axis=0)

data = {
    "t": np.repeat(np.arange(1, T + 1), J),
    "j": np.tile(np.arange(1, J + 1), T),
    "y": y.reshape(-1),
    "x": x.reshape(-1),
    "z": z.reshape(-1),
    # "eta": eta.reshape(-1),
    # "xi": xi.reshape(-1),
    "p": p.reshape(-1),
}

df = pd.DataFrame(data)
df.head()

,t,j,y,x,z,p
0,1,1,0.339222,-0.258020,-0.864535,0.177274
1,1,2,0.000470,-0.266405,1.613832,4.625309
2,2,1,0.006388,-1.099367,-0.044760,3.735276
3,2,2,0.261275,0.084040,-1.113439,0.625136
4,3,1,0.008004,-0.285171,1.246466,3.103280


### 1. Simulation draws and share function.

Fix $M=999$ draws $W_1,\dots,W_M \overset{\mathrm{i.i.d.}}{\sim} \mathcal{N}(0,1)$ once and keep them fixed. Define the deterministic mapping
$$
\alpha_m(\mu,\sigma)=\mu+\sigma W_m,\quad m=1,\dots,M,
$$
and a deterministic simulator $g_M(x_t,p_t,\xi_t;\mu,\sigma,\beta)$ that returns the model-implied inside-good shares in market $t$ given $(x_t,p_t,\xi_t)$ and $\theta$, computed by averaging logit probabilities over the $\alpha_m(\mu,\sigma)$ draws and including an outside option in the denominator.

In [3]:
M_sim = 999
W_sim = rng.normal(0.0, 1.0, size=M_sim)

def alpha_m(mu, sigma, W_m=W_sim):
    return mu + sigma * W_m

def g_M(x_t, p_t, xi_t, mu, sigma, beta):
    """
    Simulates inside shares.
    """
    x_arr = np.asarray(x_t)
    p_arr = np.asarray(p_t)
    xi_arr = np.asarray(xi_t)

    v = beta * x_arr + xi_arr
    alpha_sim = alpha_m(mu, sigma)
    U = v[None, :, :] - alpha_sim[:, None, None] * p_arr[None, :, :]

    denom = 1.0 + np.exp(logsumexp(U, axis=2))  # (M_sim, T)
    probs = np.exp(U) / denom[:, :, None]       # (M_sim, T, J)
    shares = probs.mean(axis=0)                 # (T, J)

    return shares

### 2. BLP share inversion (contraction mapping).

For a given $\theta$, implement the BLP fixed-point algorithm that inverts observed shares to recover $\xi(\theta)$:
$$
\xi^{(s)}_{jt} \leftarrow \xi^{(s-1)}_{jt} + \log y_{jt} - \log g_{M,j}(x_t,p_t,\xi^{(s-1)}_t;\theta), \qquad j=1,2.
$$
Use a convergence threshold of $10^{-8}$, and allow a generous iteration cap.

In [4]:
def invert_xi(theta, y, x, p, xi_init=None, tol=1e-8, max_iter=10000):
    """
    BLP contraction mapping to recover xi(theta).
    """
    mu, sigma, beta = theta
    T, J = y.shape

    if xi_init is None:
        xi = np.zeros((T, J))
    else:
        xi = np.array(xi_init, copy=True)

    log_y = np.log(np.clip(y, 1e-300, None))

    for it in range(max_iter):
        g = g_M(x, p, xi, mu, sigma, beta)
        g = np.clip(g, 1e-300, None)
        xi_new = xi + (log_y - np.log(g))

        diff = np.max(np.abs(xi_new - xi))
        xi = xi_new
        if diff < tol:
            return xi, it + 1, diff

    raise RuntimeError(f"Contraction did not converge in {max_iter} iterations. Final diff={diff:.3e}")

### 3. Identity-weighted GMM.

Define the instrument vector (common across the two product equations) as
$$
V_t=[1, z_{1t}, z_{2t}, x_{1t}, x_{2t}
]^\intercal
$$
Using $\xi(\theta)$ obtained from the inversion, form moments
$$
\mathbb{E}[\xi_{jt}(\theta)\,V_t]=0,\qquad j=1,2,
$$
and use the identity weighting matrix so the sample criterion is the squared Euclidean norm of the stacked sample moments.

In [5]:
def stacked_moments(xi_hat, x, z):
    """
    Returns stacked sample moments:
      [E[xi_1t * V_t], E[xi_2t * V_t]]
    where V_t = [1, z_1t, z_2t, x_1t, x_2t].
    """
    T = xi_hat.shape[0]
    V = np.column_stack([
        np.ones(T),
        z[:, 0], z[:, 1],
        x[:, 0], x[:, 1],
    ])  # (T, 5)

    m1 = (xi_hat[:, [0]] * V).mean(axis=0)  # (5,)
    m2 = (xi_hat[:, [1]] * V).mean(axis=0)  # (5,)
    return np.concatenate([m1, m2])  # (10,)


def make_gmm_objective(y, x, p, z, tol=1e-8, max_iter=10000):
    """Identity-weighted GMM objective with warm-started xi."""
    cache = {"xi": np.zeros_like(y), "last_iter": None, "last_diff": None}

    def obj(theta):
        mu, sigma, beta = theta

        # enforce sigma > 0 with a smooth penalty
        if sigma <= 0:
            return 1e8 + (1 - sigma) ** 2

        xi_hat, n_iter, diff = invert_xi(
            theta=theta,
            y=y,
            x=x,
            p=p,
            xi_init=cache["xi"],
            tol=tol,
            max_iter=max_iter,
        )

        cache["xi"] = xi_hat
        cache["last_iter"] = n_iter
        cache["last_diff"] = diff

        m = stacked_moments(xi_hat, x=x, z=z)
        return float(m @ m)  # identity weighting

    obj.cache = cache
    return obj


def gmm_estimator(theta_init, y, x, p, z):
    obj = make_gmm_objective(y=y, x=x, p=p, z=z)

    res = minimize(
        obj,
        x0=np.array(theta_init, dtype=float),
        method="L-BFGS-B",
        bounds=[(None, None), (1e-6, None), (None, None)],
        options={"maxiter": 400},
    )

    res.last_contraction_iter = obj.cache["last_iter"]
    res.last_contraction_diff = obj.cache["last_diff"]
    return res

### 4. Numerical optimization

Numerically minimize the sample GMM objective over $\theta=(\mu,\sigma,\beta)$ subject to $\sigma>0$. Run the optimizer from multiple starting values (including one at the true parameter), and report for each run the estimate $\hat\theta$ and the attained objective value.

Note: Since every optimization step runs the inversion, it's much more efficient to warm start the inversion at the converged $\xi$ of the last optimization step.

In [6]:
Y = df["y"].to_numpy().reshape(T, J)
X = df["x"].to_numpy().reshape(T, J)
Z = df["z"].to_numpy().reshape(T, J)
P = df["p"].to_numpy().reshape(T, J)

start_values = [
    np.array([2.0, 0.5, 0.5]),   # true parameter
    np.array([1.0, 0.3, 0.2]),
    np.array([3.0, 1.0, 1.0]),
    np.array([0.5, 0.8, -0.2]),
    np.array([2.5, 0.1, 1.2]),
]

rows = []
for k, theta0 in enumerate(start_values, start=1):
    print(f"Starting run {k} with theta0 = {theta0}")
    res = gmm_estimator(theta_init=theta0, y=Y, x=X, p=P, z=Z)

    rows.append(
        {
            "run": k,
            "theta0_mu": theta0[0],
            "theta0_sigma": theta0[1],
            "theta0_beta": theta0[2],
            "mu_hat": res.x[0],
            "sigma_hat": res.x[1],
            "beta_hat": res.x[2],
            "objective": res.fun,
            "success": bool(res.success),
            "message": str(res.message),
            "opt_iter": int(res.nit) if hasattr(res, "nit") else np.nan,
            "fn_evals": int(res.nfev) if hasattr(res, "nfev") else np.nan,
            "last_contraction_iter": res.last_contraction_iter,
            "last_contraction_diff": res.last_contraction_diff,
        }
    )

    print(
        f"Finished run {k}: mu={res.x[0]:.3f}, sigma={res.x[1]:.3f}, "
        f"beta={res.x[2]:.3f}, objective={res.fun:.4g}, success={res.success}"
    )

results = pd.DataFrame(rows)
results = results.sort_values("objective").reset_index(drop=True)
results

Starting run 1 with theta0 = [2.  0.5 0.5]
Finished run 1: mu=1.860, sigma=0.370, beta=0.467, objective=0.0109, success=True
Starting run 2 with theta0 = [1.  0.3 0.2]
Finished run 2: mu=1.859, sigma=0.369, beta=0.466, objective=0.0109, success=True
Starting run 3 with theta0 = [3. 1. 1.]
Finished run 3: mu=1.858, sigma=0.369, beta=0.466, objective=0.0109, success=True
Starting run 4 with theta0 = [ 0.5  0.8 -0.2]
Finished run 4: mu=1.859, sigma=0.369, beta=0.466, objective=0.0109, success=True
Starting run 5 with theta0 = [2.5 0.1 1.2]
Finished run 5: mu=1.857, sigma=0.369, beta=0.466, objective=0.0109, success=True


,run,theta0_mu,theta0_sigma,theta0_beta,mu_hat,sigma_hat,beta_hat,objective,success,message,opt_iter,fn_evals,last_contraction_iter,last_contraction_diff
0,4,0.5,0.8,-0.2,1.858929,0.369369,0.466163,0.010897,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,10,100,5,5.203701e-09
1,2,1.0,0.3,0.2,1.858842,0.369308,0.466269,0.010897,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,12,100,5,5.199757e-09
2,3,3.0,1.0,1.0,1.858382,0.369161,0.466049,0.010897,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,14,104,5,5.190230e-09
3,1,2.0,0.5,0.5,1.859860,0.369605,0.466968,0.010899,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,11,92,5,5.219032e-09
4,5,2.5,0.1,1.2,1.857376,0.368836,0.465524,0.010900,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,11,84,5,5.169102e-09
